# Data Loading Tutorial

This tutorial covers how to load EEG data from various formats supported by NeuRodent.

## Overview

NeuRodent supports multiple data formats commonly used in rodent EEG research:

1. **Binary files** (`.bin`) - Custom binary format
2. **SpikeInterface recordings** - Via the SpikeInterface library
3. **MNE objects** - From the MNE-Python library
4. **Neuroscope/Neuralynx** (`.dat`, `.eeg`)
5. **Open Ephys** (`.continuous`)
6. **NWB files** (`.nwb`) - Neurodata Without Borders format

The `LongRecordingOrganizer` class handles loading and organizing recordings from these formats.

## Setup

In [ ]:
import sys
from pathlib import Path
import logging

from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt

from neurodent import core
from neurodent import constants

import mne
import spikeinterface.core as si
import spikeinterface.extractors as se


# Set up logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(message)s", 
    level=logging.INFO
)
logger = logging.getLogger()

## 1. Loading Binary Files

Binary files are a common format for storing continuous EEG data. NeuRodent can load binary files with associated metadata.

In [ ]:
# Example: Loading from binary files
# Using included test data
data_path = Path("../../notebooks/tests/test-data/F22 KO 12_12_2023")
animal_id = "F22 KO 12_12_2023"

# Create LongRecordingOrganizer
# mode options: 'bin', 'si' (SpikeInterface), 'mne', etc.
lro_bin = core.LongRecordingOrganizer(
    base_folder_path=data_path,
    animal_id=animal_id,
    mode="bin",  # Change based on your data format
)

print(f"Loaded recordings for {animal_id}")
print(f"Number of recordings: {len(lro_bin.colbins)}")
print(f"Sampling frequency: {lro_bin.meta.f_s}")
print(f"Number of channels: {lro_bin.meta.n_channels}")

In [ ]:
lro_mne = lro.convert_to_mne()

for i in range(len(lro_mne.info['ch_names'])):
    mne_chname = lro_mne.info['ch_names'][i]
    lro_mne.info['ch_names'][i] = core.utils.parse_chname_to_abbrev(channel_name = mne_chname, assume_from_number=True, strict_matching=False)

#mne.export.export_raw("../../notebooks/tests/test-data/F22 KO 12_12_2023/F22_si.edf", lro_mne, fmt="edf")
lro_mne.save("../../notebooks/tests/test-data/A10 KO 12_13_2023/A10_mne.fif", overwrite=True)


### Binary File Format Details

NeuRodent supports two binary layouts:

1. **Column-major** (default): Each file contains data for one channel
2. **Row-major**: Each file contains all channels, with samples interleaved

You can convert between formats:

In [ ]:
meta_path = "../../notebooks/tests/test-data/A10 KO 12_13_2023/Cage 2 A10-0_Meta.csv"
metadata = core.DDFBinaryMetadata(metadata_path=meta_path)

# Convert column-major to row-major format
col_path = "../../notebooks/tests/test-data/A10 KO 12_13_2023/Cage 2 A10-0_ColMajor.bin"
row_out_path = "../../notebooks/tests/test-data/A10 KO 12_13_2023"

core.convert_ddfcolbin_to_ddfrowbin(
    colbin_path=col_path,
    rowdir_path=row_out_path,
    metadata = metadata
)

# Convert row-major file to SpikeInterface format
# The output will be tuple with SpikeInterface recording and temporary file path
row_path = "../../notebooks/tests/test-data/A10 KO 12_13_2023/Cage 2 A10-0_RowMajor.npy.gz"
conv_si = core.convert_ddfrowbin_to_si(
    bin_rowmajor_path=row_path,
    metadata = metadata
)

# Extract SpikeInterface recording object and print information
rec_si = conv_si[0]
print(f"Sampling frequency: {rec_si.sampling_frequency}")
print(f"Number of channels: {rec_si.get_num_channels()}")

## 2. Loading SpikeInterface Recordings

SpikeInterface is a popular Python library for extracellular electrophysiology data. NeuRodent can directly use SpikeInterface recordings:

In [ ]:
# need pyedflig

# Example: Loading SpikeInterface recordings
si_data_path = Path("../../notebooks/tests/test-data/A10 KO 12_13_2023")

lro_si = core.LongRecordingOrganizer(
    base_folder_path=si_data_path,
    mode="si",  # Change based on your data format
    manual_datetimes=datetime(2023, 12, 12),
    extract_func=se.read_edf,
    stream_id='0',
    input_type='file',
    file_pattern='*.edf',
)

print(f"Number of recordings: {len(lro.colbins)}")
print(f"Sampling frequency: {lro.meta.f_s}")
print(f"Number of channels: {lro.meta.n_channels}")

## 3. Loading MNE Objects

MNE-Python is a widely-used library for MEG and EEG analysis. NeuRodent can work with MNE Raw objects:

In [ ]:
import mne
from datetime import datetime
from pathlib import Path

data_path = Path("../../notebooks/tests/test-data/A10 KO 12_13_2023")
animal_id = "A10 KO 12_13_2023"


# Create LongRecordingOrganizer with MNE object
lro_mne = core.LongRecordingOrganizer(
    base_folder_path=data_path,
    manual_datetimes=datetime(2023, 12, 13),
    mode="mne",
    extract_func = mne.io.read_raw_fif,
    input_type='file',
    file_pattern = '*.fif',
    
)

print(f"Sampling frequency: {lro_mne.meta.f_s}")
print(f"Number of channels: {lro_mne.meta.n_channels}")

## 4. Inspecting Loaded Data

Once data is loaded, you can inspect its properties. Here, we use LRO object from binary file (Section 1):

In [ ]:
# Get basic properties using metadata from lro object

metadata = lro_bin.meta

print(f"Recording metadata: {metadata}")

print(f"Sampling frequency: {metadata.f_s} Hz")
print(f"Number of channels: {metadata.n_channels}")
print(f"Channel names: {metadata.channel_names}")
print(f"Units: {metadata.V_units}")


print(f"Duration: {lro_bin.file_durations} seconds")

## 5. Loading Other Formats

Neurodent supports other data formats as well, such as Neuroscope/Neuralyns, Ephys, NWB files, etc.
Although the example datasets are not provided, sections below serve as a starting point for loading these formats.

### Neuroscope/Neuralynx

For `.dat` or `.eeg` files:

In [ ]:
# Load using SpikeInterface extractors
neuroscope_path = Path("/path/to/neuroscope/data.dat")
recording_neuroscope = se.read_neuroscope(neuroscope_path)

lro_neuroscope = core.LongRecordingOrganizer(
    base_folder=None,
    animal_id=animal_id,
    mode="si",
    si_recordings=[recording_neuroscope],
)

### Open Ephys

For Open Ephys `.continuous` files:

In [ ]:
# Load using SpikeInterface extractors
openephys_path = Path("/path/to/openephys/folder")
recording_openephys = se.read_openephys(openephys_path)

lro_openephys = core.LongRecordingOrganizer(
    base_folder=None,
    animal_id=animal_id,
    mode="si",
    si_recordings=[recording_openephys],
)

### NWB Files

Neurodata Without Borders (NWB) is a standardized format for neurophysiology data:

In [ ]:
# Example: Loading NWB files
nwb_path = Path("/path/to/nwb/file.nwb")

# First, load with SpikeInterface's NWB extractor
import spikeinterface.extractors as se

recording_nwb = se.read_nwb(nwb_path)

# Then use with LongRecordingOrganizer
lro_nwb = core.LongRecordingOrganizer(
    base_folder=None,
    animal_id=animal_id,
    mode="si",
    si_recordings=[recording_nwb],
)

print(f"Loaded NWB data with {len(lro_nwb.recordings)} recordings")

## 6. Working with Multiple Recordings

NeuRodent can handle multiple recordings from the same animal (e.g., different sessions or days):

In [ ]:
# Example: Loading multiple recordings
data_folder = Path("/path/to/multi/session/data")

lro_multi = core.LongRecordingOrganizer(
    base_folder=data_folder,
    animal_id=animal_id,
    mode="bin",
)

print(f"Total recordings: {len(lro_multi.recordings)}")

# Iterate through recordings
for i, recording in enumerate(lro_multi.recordings):
    duration = recording.get_num_frames() / recording.get_sampling_frequency()
    print(f"Recording {i}: {duration:.1f} seconds")

## 7. Advanced: Custom Data Loading

For custom formats, you can create SpikeInterface Recording objects and pass them to `LongRecordingOrganizer`:

In [ ]:
import spikeinterface as si

# Example: Create a recording from numpy array
# (useful for custom formats or testing)
num_channels = 16
sampling_frequency = 1000  # Hz
duration = 60  # seconds
num_samples = int(sampling_frequency * duration)

# Generate random data (replace with your actual data)
data = np.random.randn(num_channels, num_samples)

# Create SpikeInterface recording
recording_custom = si.NumpyRecording(
    traces_list=[data],
    sampling_frequency=sampling_frequency,
)

# Set channel IDs
channel_ids = [f"CH{i:02d}" for i in range(num_channels)]
recording_custom = recording_custom.rename_channels(
    new_channel_ids=channel_ids
)

# Use with LongRecordingOrganizer
lro_custom = core.LongRecordingOrganizer(
    base_folder=None,
    animal_id=animal_id,
    mode="si",
    si_recordings=[recording_custom],
)

print("Custom recording created successfully!")

## Summary

In this tutorial, you learned:

1. How to load data from multiple formats (binary, SpikeInterface, MNE, NWB, etc.)
2. How to inspect loaded data properties
3. How to handle metadata and timing information
4. How to work with multiple recordings
5. How to create custom recordings for non-standard formats

## Next Steps

- **[Basic Usage Tutorial](basic_usage.ipynb)**: Complete workflow from loading to visualization
- **[Windowed Analysis Tutorial](../tutorials/windowed_analysis.ipynb)**: Extract features from loaded data
- **[Spike Analysis Tutorial](../tutorials/spike_analysis.ipynb)**: Work with spike-sorted data